# Qwen3-Embedding-0.6B — DIMER E2E contrastive fine-tuning tutorial: intent retrieval on Banking77 (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/qwen3-embedding-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/qwen3-embedding-pipeline/blob/main/tutorials/qwen3_embedding_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Qwen%2FQwen3--Embedding--0.6B-ffcc4d?style=flat)](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) [![Upstream](https://img.shields.io/badge/Upstream-QwenLM%2FQwen3--Embedding-181717?style=flat&logo=github&logoColor=white)](https://github.com/QwenLM/Qwen3-Embedding) [![arXiv](https://img.shields.io/badge/arXiv-2506.05176-b31b1b.svg)](https://arxiv.org/abs/2506.05176)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** text embeddings (1024-d, last-token pooled, L2-normalised, instruction-aware queries) and bounded contrastive fine-tuning of the last decoder layers on query–positive pairs, measured by held-out retrieval recall@k and MRR, using the pinned `Qwen/Qwen3-Embedding-0.6B` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/qwen3_embedding_pipeline/`, at revision `9438a7b6cce5`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3` (~1207 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned Qwen3-Embedding-0.6B snapshot (safetensors, 1.19 GB), fetches the two digest-pinned Banking77 CSV files from the project repository (1.1 MB, no credential), pairs every customer message with its intent phrase and draws 616 / 154 / 385 training, validation and test pairs balanced over the 77 intents from the release's own partition, embeds three test queries and the 77 intent documents through the inference contract with an input manifest and a rejection probe, scores the frozen embedder on the test queries by recall@1 / recall@5 / recall@10 and MRR over the 77 documents beside the random floor and a lexical (token-overlap) baseline, runs a bounded contrastive fine-tuning (InfoNCE with in-batch negatives) of the last two decoder layers with validation-MRR epoch selection, scores the held-out split again, retrieves for the same three queries with the adapted embedder, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about four minutes of model time after the downloads; a CUDA runtime is used automatically when present (bfloat16 there, float32 on CPU).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own query–positive pairs as a CSV (columns `id`, `query`, `positive`, optional `negative`), a JSON array or a JSONL file of `{{id, query, positive}}` records — the document set is the unique positives (and negatives); set `INSTRUCTION` to a one-line description of your retrieval task. They pass through the same validation, seeded query-disjoint split, floor and baseline, frozen scoring, fine-tuning, held-out evaluation, retrieval, artifact export and reload-parity cells as the Banking77 sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference each text is tokenised with left padding and truncated at 8,192 tokens, a 0.6 B-parameter Qwen3 decoder encodes it, the hidden state of the **last token** is taken as the text's vector, and the pipeline L2-normalises it to unit length. Queries are prefixed with a task instruction (`Instruct: …\nQuery:`) because the model is instruction-aware; documents are embedded as-is. **Embeddings are representations, not predictions:** a vector carries no score, and the carried pipeline module adds snapshot verification, input validation with named ceilings, the query/document formatting contract, a fixed output contract and the `cosine_similarity`, `validate_inputs` and `evaluation_report` helpers.

What this notebook adds to inference is **adaptation measured by retrieval**. The dataset is real: Banking77 (Casanueva et al., 2020; CC BY 4.0) ships 13,083 customer-support messages labelled with 77 fine-grained banking intents as two digest-pinned CSV files fetched from the project repository at a pinned commit. Every intent name becomes a short **document** (`card_arrival` → `card arrival`) and every message a **query** whose positive is its intent phrase, so nearest-neighbour retrieval over the 77 documents is intent detection — a task the embedder was never tuned for, with documents that are two or three words long. The carried `metrics.py` ranks the documents for every held-out query by cosine and reads the rank of its positive: **recall@1**, **recall@5**, **recall@10** and **MRR**; a **random floor** (1 / 77 recall@1) and a **lexical baseline** (Jaccard token overlap between message and phrase) frame the frozen number. The fine-tuning question is whether a bounded contrastive adaptation of the last decoder layers on 616 pairs raises retrieval on messages the model has not seen. Nothing here is a quality claim about your retrieval task: it is one seeded split of one corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned real corpus and validate and split it without leakage; embed queries and documents through the public API with the instruction contract and read the vector contract correctly; read recall@k and MRR beside a random floor and a lexical baseline and understand what they do and do not measure; run a bounded contrastive fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; compare retrieved documents before and after; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** reranking (see the sibling Qwen3 reranker pipeline), text generation or chat, classification heads, clustering quality, Matryoshka dimension truncation (fixed at 1024 here), hard-negative mining beyond the in-batch and explicit negatives of the pair contract, full-model or embedding-table training, and any claim that a Banking77 intent split stands in for your retrieval task. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available. **Precision differs by device:** the pipeline runs float32 on CPU and bfloat16 on CUDA, so cosine values and the recorded metrics can differ between the two. CPU is adequate for this sample: the build record measured about 5 s to load and digest-verify the 1.19 GB snapshot, about 30 s to embed the 385 test queries and 77 documents, and about 47 s per training epoch over 616 pairs plus a validation pass per epoch. The pinned `torch==2.14.0` install and the 1.19 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and NumPy; what a dense vector, a unit norm and cosine similarity are; what recall@k and mean reciprocal rank measure and why they need a labelled query–document set; what a contrastive (InfoNCE) loss with in-batch negatives does.
- **Data contract:** records are `{{id, query, positive}}` — a query of 1..100,000 characters (text beyond 8,192 tokens is truncated and flagged at inference; queries and documents are truncated to 64 tokens **during training only**), a positive document of 1..1,000 characters, an optional `negative` document (added to the document set), ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records and at least 2 distinct positives; queries are de-duplicated case-insensitively before splitting so the same message never sits in two splits. BYOD accepts CSV, JSON or JSONL in that shape.
- **Validation is structural, not semantic:** nothing checks that a positive is relevant to its query or that the instruction describes the task — a mislabelled pair set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal query log with its relevance labels is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches two pinned objects (`train.csv` 839,073 bytes, `test.csv` 239,961 bytes; SHA-256 `b06e26ac…` / `d12d6e3b…`) from `raw.githubusercontent.com` at the pinned `PolyAI-LDN/task-specific-datasets` commit over HTTPS, each refused on any mismatch before it is read; Banking77 is CC BY 4.0 (Casanueva et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `Qwen/Qwen3-Embedding-0.6B` snapshot (~1207 MB in total) at revision `97b0c614be4d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'qwen3-embedding-pipeline',
    'repository_revision': '9438a7b6cce52f32db1e035e7c2b5b3eabf6bf28',
    'embedded_module': 'src/qwen3_embedding_pipeline/pipeline.py',
    'embedded_modules': ['src/qwen3_embedding_pipeline/metrics.py', 'src/qwen3_embedding_pipeline/pipeline.py', 'src/qwen3_embedding_pipeline/samples.py'],
    'module_sha256': 'ccdf813ff7fb81873947978122e4402cf2ec37591546cc4b7d2ac61bbef7a4d0',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/qwen3_embedding_pipeline/` @ `9438a7b6cce5`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/qwen3_embedding_pipeline/metrics.py`

In [ ]:
"""Retrieval metrics over a query–positive dataset and a lexical baseline.

Every query is ranked against the dataset's document set (its sorted unique positives) by a similarity the
caller supplies — the embedder's cosine, or bag-of-words overlap for the baseline — and the rank of the
query's own positive is read: **recall@1**, **recall@5** and **recall@10** (the positive is within the first
*k*), and **MRR** (mean of 1 / rank). Ties are resolved pessimistically (a tied document counts as ranked
above the positive), so a similarity that cannot separate documents scores as badly as it deserves. The
**random floor** is what a uniformly random ranking over the document set achieves in expectation; the
**lexical baseline** ranks documents by the Jaccard overlap of lower-cased alphanumeric tokens with the
query — what a system with no model at all gets from shared words.
"""

from __future__ import annotations

import re
from collections.abc import Mapping, Sequence
from typing import Any

RECALL_AT = (1, 5, 10)
METRIC_DEFINITIONS = {
    "recall@k": (
        "fraction of queries whose positive document is ranked within the first k of the document set; "
        "ties count against the positive"
    ),
    "mrr": "mean over queries of 1 / rank of the positive document",
    "document_set": "the sorted unique positive (and explicit negative) documents of the scored dataset",
}
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def rank_of_positive(scores: Sequence[float], positive_index: int) -> int:
    """1-based rank of `positive_index` under descending `scores`; ties are ranked above the positive."""
    if not 0 <= positive_index < len(scores):
        raise ValueError("positive_index is outside the document set")
    target = float(scores[positive_index])
    return 1 + sum(1 for i, s in enumerate(scores) if i != positive_index and float(s) >= target)


def retrieval_metrics(ranks: Sequence[int], n_documents: int) -> dict[str, Any]:
    """Aggregate 1-based ranks of each query's positive into recall@k and MRR."""
    if not ranks:
        raise ValueError("no queries to score")
    if n_documents < 1:
        raise ValueError("n_documents must be positive")
    if any(not isinstance(r, int) or r < 1 or r > n_documents for r in ranks):
        raise ValueError("every rank must be an int in 1..n_documents")
    out: dict[str, Any] = {"n_queries": len(ranks), "n_documents": n_documents}
    for k in RECALL_AT:
        out[f"recall@{k}"] = sum(1 for r in ranks if r <= k) / len(ranks)
    out["mrr"] = sum(1.0 / r for r in ranks) / len(ranks)
    out["median_rank"] = sorted(ranks)[len(ranks) // 2]
    out["definitions"] = dict(METRIC_DEFINITIONS)
    return out


def random_floor(n_documents: int) -> dict[str, Any]:
    """Expected metrics of a uniformly random ranking over `n_documents` documents."""
    if n_documents < 1:
        raise ValueError("n_documents must be positive")
    out: dict[str, Any] = {"n_documents": n_documents}
    for k in RECALL_AT:
        out[f"recall@{k}"] = min(k, n_documents) / n_documents
    out["mrr"] = sum(1.0 / r for r in range(1, n_documents + 1)) / n_documents
    out["baseline"] = "uniformly random ranking of the document set (expected values)"
    return out


def _tokens(text: str) -> set[str]:
    return set(_TOKEN_RE.findall(text.lower()))


def jaccard(a: str, b: str) -> float:
    x, y = _tokens(a), _tokens(b)
    if not x or not y:
        return 0.0
    return len(x & y) / len(x | y)


def lexical_baseline(records: Sequence[Mapping[str, Any]], documents: Sequence[str]) -> dict[str, Any]:
    """Rank the documents for every query by Jaccard token overlap — the no-model floor."""
    index = {doc: i for i, doc in enumerate(documents)}
    ranks = []
    for record in records:
        if record["positive"] not in index:
            raise ValueError(f"positive {record['positive']!r} is not in the document set")
        scores = [jaccard(record["query"], doc) for doc in documents]
        ranks.append(rank_of_positive(scores, index[record["positive"]]))
    result = retrieval_metrics(ranks, len(documents))
    result["baseline"] = "Jaccard overlap of lower-cased alphanumeric tokens between query and document"
    return result

**Module 2/3:** `src/qwen3_embedding_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
MODEL_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "qwen3-embedding-0.6b"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Contract from the pinned upstream README ("Transformers Usage"): left padding, last-token pooling,
# L2 normalisation, max_length 8192, and an "Instruct: ...\nQuery:" prefix on queries only.
EMBEDDING_DIM = 1024  # hidden_size in the pinned config.json; 1_Pooling/config.json word_embedding_dimension
MAX_TEXT_TOKENS = 8192  # tokenizer truncation length; the model's context is 32768 but the README uses 8192
MAX_TEXT_CHARS = 100_000  # pre-tokenisation guard so a runaway string is rejected before it is tokenised
MAX_BATCH = 64  # texts per embed() call
DEFAULT_QUERY_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
POOLING = "last_token"
KINDS = ("query", "document")
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 595_776_512  # Qwen3Model (no LM head)
DECODER_LAYERS = 28  # config.json num_hidden_layers
DEFAULT_TRAINABLE_LAYERS = 2  # the last two decoder layers (31,461,888 parameters)
MAX_TRAIN_TOKENS = (
    64  # training-only truncation of queries and documents (inference truncates at MAX_TEXT_TOKENS)
)
DEFAULT_TEMPERATURE = 0.05
MAX_EVAL_RECORDS = 2_000
MAX_DOCUMENTS = 1_000
MIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.qwen3-embedding-0.6b.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_query(query: str, instruction: str = DEFAULT_QUERY_INSTRUCTION) -> str:
    """Upstream `get_detailed_instruct`: queries carry a task instruction, documents do not."""
    return f"Instruct: {instruction}\nQuery:{query}"


def cosine_similarity(a: Sequence[Sequence[float]], b: Sequence[Sequence[float]]) -> list[list[float]]:
    """Cosine similarity matrix between two lists of vectors (no metric: there is no ground truth)."""
    x = np.asarray(a, dtype=np.float32)
    y = np.asarray(b, dtype=np.float32)
    if x.ndim != 2 or y.ndim != 2 or x.shape[1] != y.shape[1]:
        raise ValueError("inputs must be 2-D with the same embedding dimension")
    x = x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    y = y / np.maximum(np.linalg.norm(y, axis=1, keepdims=True), 1e-12)
    return (x @ y.T).tolist()


INPUT_SCHEMA: dict[str, Any] = {
    "input": "sequence of non-empty str; one vector is returned per text, in input order",
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "text_tokens": [1, MAX_TEXT_TOKENS],
    "kind": list(KINDS),
    "embedding_dim": EMBEDDING_DIM,
    "preprocessing": (
        "left-padded tokenisation truncated at MAX_TEXT_TOKENS; kind='query' prepends "
        "'Instruct: <instruction>\\nQuery:'; last-token pooling, then L2 normalisation"
    ),
}


def _check_inputs(texts: Any, kind: str, instruction: str) -> list[str]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the texts as a list."""
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..{MAX_BATCH} items, got {len(texts)}")
    for i, text in enumerate(texts):
        if not isinstance(text, str):
            raise TypeError(f"texts[{i}] must be str, got {type(text).__name__}")
        if not text.strip():
            raise ValueError(f"texts[{i}] is empty")
        if len(text) > MAX_TEXT_CHARS:
            raise ValueError(f"texts[{i}] has {len(text)} chars; ceiling is {MAX_TEXT_CHARS}")
    if kind not in KINDS:
        raise ValueError(f"kind must be one of {KINDS}")
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError("instruction must be a non-empty str")
    return list(texts)


def validate_inputs(
    texts: Sequence[str],
    kind: str = "document",
    instruction: str = DEFAULT_QUERY_INSTRUCTION,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``embed`` would — both route through
    ``_check_inputs`` — so a caller that wants the finding recorded catches the exception and
    stores ``str(exc)`` under ``findings``. Token-level truncation cannot be observed here
    because it happens inside the tokenizer; ``embed`` reports it in ``truncated``.
    """
    checked = _check_inputs(texts, kind, instruction)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"text-{i}", "chars": len(text), "kind": kind}
            for i, text in enumerate(checked)
        ],
        "kind": kind,
        "instruction": instruction if kind == "query" else None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], labels: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Embeddings are representations, so the repository ships no performance metric —
    ``cosine_similarity`` is a comparison helper, not a score against ground truth. The verdict
    is therefore always ``not-measurable`` (EVAL9), including when ``labels`` is supplied:
    the parameter exists for interface parity with the fleet's other pipelines and is recorded
    in ``reason`` rather than scored.
    """
    embeddings = result["embeddings"]
    supplied = labels is not None
    return {
        "task": "text embedding (dense representation, no label space)",
        "score_semantics": (
            f"{EMBEDDING_DIM}-d unit-norm vectors, {POOLING} pooling; cosine between two vectors of "
            "this model is a similarity in [-1, 1], not a probability and not calibrated"
        ),
        "sample_kind": sample_kind,
        "n_texts": len(embeddings),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the output is a representation, not a prediction: the pipeline exposes no performance "
            "metric, only the cosine_similarity comparison helper"
            + ("; labels were supplied but no metric helper exists to score them here" if supplied else "")
        ),
        "needs": (
            "a downstream labelled task: for retrieval, a query-document set with relevance "
            "judgements scored by nDCG@k or recall@k; for classification or clustering, labelled "
            "texts and a fitted classifier or cluster assignment — none of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class Qwen3EmbeddingPipeline:
    """Text embedder. `_runner` maps formatted texts to (pooled un-normalised vectors, token counts)."""

    _runner: Callable[[list[str]], tuple[np.ndarray, list[int]]]
    device: str
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Qwen3EmbeddingPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), dict(local_files_only=True)
        elif allow_download:
            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoModel, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(
            source, padding_side="left", trust_remote_code=False, **kwargs
        )
        model = AutoModel.from_pretrained(source, dtype=dtype, trust_remote_code=False, **kwargs)
        model = model.to(resolved_device).eval()

        def runner(texts: list[str]) -> tuple[np.ndarray, list[int]]:
            batch = tokenizer(
                texts, padding=True, truncation=True, max_length=MAX_TEXT_TOKENS, return_tensors="pt"
            )
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                hidden = model(**batch).last_hidden_state
            pooled = hidden[:, -1]  # left padding: the last position is the last real token of every row
            counts = batch["attention_mask"].sum(dim=1).tolist()
            return pooled.float().cpu().numpy(), [int(c) for c in counts]

        return cls(runner, resolved_device, _model=model, _tokenizer=tokenizer)

    def _validate(self, texts: Any, kind: str, instruction: str) -> list[str]:
        return _check_inputs(texts, kind, instruction)

    def embed(
        self,
        texts: Sequence[str],
        kind: str = "document",
        instruction: str = DEFAULT_QUERY_INSTRUCTION,
    ) -> dict[str, Any]:
        """Embed up to MAX_BATCH texts. `kind="query"` prepends the instruction; documents get none."""
        texts = self._validate(texts, kind, instruction)
        formatted = [format_query(t, instruction) if kind == "query" else t for t in texts]
        pooled, n_tokens = self._runner(formatted)
        pooled = np.asarray(pooled, dtype=np.float32)
        if pooled.shape != (len(texts), EMBEDDING_DIM):
            raise RuntimeError(f"backend returned {pooled.shape}, expected ({len(texts)}, {EMBEDDING_DIM})")
        normalized = pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)
        return {
            "embeddings": normalized.tolist(),
            "dim": EMBEDDING_DIM,
            "pooling": POOLING,
            "normalized": True,
            "kind": kind,
            "instruction": instruction if kind == "query" else None,
            "n_tokens": list(n_tokens),
            "truncated": [n >= MAX_TEXT_TOKENS for n in n_tokens],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def _embed_all(self, texts: Sequence[str], kind: str, instruction: str) -> np.ndarray:
        """Embed any number of texts through the public contract, MAX_BATCH at a time."""
        rows = []
        for start in range(0, len(texts), MAX_BATCH):
            rows.extend(
                self.embed(list(texts[start : start + MAX_BATCH]), kind=kind, instruction=instruction)[
                    "embeddings"
                ]
            )
        return np.asarray(rows, dtype=np.float32)

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        instruction: str = DEFAULT_QUERY_INSTRUCTION,
        candidates: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        """Retrieval over the dataset's document set: every query (embedded with `instruction`) is ranked
        against every document by cosine and the rank of its own positive is read (recall@k, MRR)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import rank_of_positive, retrieval_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import documents, validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        docs = list(candidates) if candidates is not None else documents(checked)
        if not 2 <= len(docs) <= MAX_DOCUMENTS:
            raise ValueError(f"the document set must hold 2..{MAX_DOCUMENTS} documents; got {len(docs)}")
        index = {doc: i for i, doc in enumerate(docs)}
        missing = [r["positive"] for r in checked if r["positive"] not in index]
        if missing:
            raise ValueError(f"positive {missing[0]!r} is not in the document set")
        started = time.perf_counter()
        doc_vectors = self._embed_all(docs, "document", instruction)
        query_vectors = self._embed_all([r["query"] for r in checked], "query", instruction)
        scores = query_vectors @ doc_vectors.T
        ranks = [
            rank_of_positive(row.tolist(), index[r["positive"]])
            for row, r in zip(scores, checked, strict=True)
        ]
        metrics = retrieval_metrics(ranks, len(docs))
        metrics.update(
            {
                "instruction": instruction,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    @staticmethod
    def lexical_baseline(
        records: Sequence[Mapping[str, Any]], candidates: Sequence[str] | None = None
    ) -> dict[str, Any]:
        """The no-model floor: documents ranked by token overlap with the query (see metrics.py)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import lexical_baseline` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import documents, validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        return lexical_baseline(checked, list(candidates) if candidates is not None else documents(checked))

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if not isinstance(trainable_layers, int) or not 1 <= trainable_layers <= DECODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_layers
        prefixes = tuple(f"layers.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        instruction: str = DEFAULT_QUERY_INSTRUCTION,
        epochs: int = 2,
        lr: float = 5e-5,
        batch_size: int = 16,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        temperature: float = DEFAULT_TEMPERATURE,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded contrastive fine-tuning on validated query–positive pairs.

        Only the last `trainable_layers` decoder layers train (2 by default; the token embeddings, the
        earlier layers and the final norm stay frozen). Each batch embeds its queries (formatted with
        `instruction`) and the unique documents among its positives (plus any explicit negatives) in one
        forward pass; the loss is the InfoNCE cross-entropy of every query over that batch's documents at
        `temperature` — the other queries' positives are the negatives — with AdamW at a fixed learning rate,
        gradient clipping at 1.0, seeded shuffling and no scheduler; texts are truncated to MAX_TRAIN_TOKENS
        **during training only**. Epoch 0 records the frozen model's validation retrieval metrics against
        the validation document set; the epoch with the highest validation MRR is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import documents, validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 2 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 2..64")
        if not (0.0 < temperature <= 1.0):
            raise ValueError("temperature must be in (0, 1]")
        _check_inputs(["x"], "query", instruction)
        names = self._trainable_names(trainable_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        val_docs = documents(val_checked) if val_checked else []
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            keep = ("recall@1", "recall@5", "recall@10", "mrr", "n_documents")
            return {
                k: v
                for k, v in self.evaluate(val_checked, instruction=instruction, candidates=val_docs).items()
                if k in keep
            }

        def encode(texts: list[str]) -> torch.Tensor:
            batch = tokenizer(
                texts, padding=True, truncation=True, max_length=MAX_TRAIN_TOKENS, return_tensors="pt"
            )
            hidden = model(**batch.to(device)).last_hidden_state[:, -1]
            return torch.nn.functional.normalize(hidden.float(), dim=-1)

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_mrr = entry["val"]["mrr"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    if len(batch) < 2:
                        continue
                    docs = documents(batch)
                    targets = torch.tensor(
                        [docs.index(r["positive"]) for r in batch], dtype=torch.long, device=device
                    )
                    queries = encode([format_query(r["query"], instruction) for r in batch])
                    candidates = encode(docs)
                    logits = queries @ candidates.T / temperature
                    loss = torch.nn.functional.cross_entropy(logits, targets)
                    optimiser.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / max(len(losses), 1), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["mrr"] if entry["val"] else math.inf
                if current > best_mrr or not entry["val"]:
                    best_mrr = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "objective": "InfoNCE over in-batch documents (contrastive)",
            "instruction": instruction,
            "trainable_layers": trainable_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation MRR" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "temperature": temperature,
            "max_train_tokens": MAX_TRAIN_TOKENS,
            "n_train": len(train_checked),
            "n_train_documents": len(documents(train_checked)),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder-layer tensors as safetensors with a manifest naming the base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported
        format and version, the pinned base (id, revision, weight file, digest), exactly one file entry
        named `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not
        authenticity against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int):
            raise ValueError("artifact manifest does not record an integer trainable_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("layers."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder-layer tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Qwen3EmbeddingPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/qwen3_embedding_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Query–positive pair dataset contract for contrastive adaptation of the embedder: the pinned Banking77
sample, validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and a retrieval task the embedder was not tuned for: Banking77 (Casanueva et
al., 2020; CC BY 4.0), 13,083 customer-support messages labelled with 77 fine-grained banking intents. Two CSV
files (`train.csv`, `test.csv`) are fetched from the PolyAI `task-specific-datasets` repository at a pinned
commit and refused on any byte-size or SHA-256 mismatch. Every intent name becomes a short **document**
(`card_arrival` → `card arrival`); each message is a **query** whose positive document is its intent phrase,
so retrieval over the 77 documents is intent detection by nearest neighbour. Training and validation queries
are drawn from `train.csv`, test queries from `test.csv` — the release's own partition — balanced over the
77 intents.

A record is ``{id, query, positive}`` (an optional ``negative`` is accepted and carried); the document set
of a dataset is its sorted unique positives.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Banking77 (messages → intent phrases)"
CORPUS_RELEASE = "PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b"
CORPUS_BASE_URL = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    "57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
)
CORPUS_FILES = {
    "train": ("train.csv", 839_073, "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"),
    "test": ("test.csv", 239_961, "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"),
}
CORPUS_LICENSE = "CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)"
CORPUS_ROWS = {"train": 10_003, "test": 3_080}
CORPUS_INTENTS = 77
DEFAULT_CACHE_DIR = Path("weights") / "banking77"
DEFAULT_INSTRUCTION = "Given a customer support message, retrieve the banking intent it expresses"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 616, "validation": 154, "test": 385}  # 8 / 2 / 5 per intent, balanced over 77
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MAX_DOCUMENT_CHARS = 1_000
MIN_DOCUMENTS = 2
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def intent_phrase(intent: str) -> str:
    """The document text of an intent: its snake_case name as words (`card_arrival` → `card arrival`)."""
    return " ".join(intent.strip().split("_"))


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the two pinned Banking77 CSVs (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the CSV members (columns `text`, `category`) into flat records keeping the raw intent name."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        rows = list(csv.DictReader(io.StringIO(files[split].decode("utf-8"))))
        if not rows or {"text", "category"} - set(rows[0]):
            raise ValueError(f"{split}: expected columns text and category")
        if len(rows) != CORPUS_ROWS[split]:
            raise ValueError(f"{split}: {len(rows)} rows, expected {CORPUS_ROWS[split]}")
        out[split] = [
            {"id": f"{split}-{i:05d}", "text": r["text"].strip(), "intent": r["category"].strip()}
            for i, r in enumerate(rows)
        ]
        intents = {r["intent"] for r in out[split]}
        if len(intents) != CORPUS_INTENTS:
            raise ValueError(f"{split}: {len(intents)} intents, expected {CORPUS_INTENTS}")
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Turn corpus rows into query–positive pairs; drop empty, over-long and repeated queries."""
    seen: set[str] = set()
    kept = []
    for record in records:
        query = str(record["text"]).strip()
        key = query.lower()
        if not query or key in seen or len(query) > MAX_TEXT_CHARS:
            continue
        seen.add(key)
        intent = str(record["intent"])
        kept.append({"id": record["id"], "query": query, "positive": intent_phrase(intent), "intent": intent})
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Balanced seeded draws over all 77 intents: training and validation from `train` (disjoint queries),
    test from `test`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    for name, size in sizes.items():
        if size % CORPUS_INTENTS:
            raise ValueError(f"{name} size {size} is not a multiple of the {CORPUS_INTENTS} intents")
    rng = random.Random(seed)
    pools = {"train": filter_records(corpus["train"]), "test": filter_records(corpus["test"])}
    intents = sorted({r["intent"] for r in pools["train"]})
    by_intent = {
        split: {intent: [r for r in pool if r["intent"] == intent] for intent in intents}
        for split, pool in pools.items()
    }
    for split in by_intent.values():
        for records in split.values():
            rng.shuffle(records)
    cursor = dict.fromkeys(intents, 0)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        source = "test" if name == "test" else "train"
        per_intent = size // CORPUS_INTENTS
        picked = []
        for intent in intents:
            pool = by_intent[source][intent]
            start = cursor[intent] if source == "train" else 0
            chunk = pool[start : start + per_intent]
            if len(chunk) < per_intent:
                raise ValueError(
                    f"{name}: only {len(chunk)} records available for {intent!r}, need {per_intent}"
                )
            picked.extend(chunk)
            if source == "train":
                cursor[intent] = start + per_intent
        rng.shuffle(picked)
        out[name] = [
            {"id": f"{name}-{i:04d}", "query": r["query"], "positive": r["positive"], "intent": r["intent"]}
            for i, r in enumerate(picked)
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_text(value: Any, label: str, ceiling: int) -> str:
    if not isinstance(value, str):
        raise ValueError(f"{label} must be a string")
    if not value.strip():
        raise ValueError(f"{label} is empty")
    if len(value) > ceiling:
        raise ValueError(f"{label} has {len(value)} chars; ceiling is {ceiling}")
    return value.strip()


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/query/positive")
    for key in ("id", "query", "positive"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    item = {
        "id": rid,
        "query": _check_text(record["query"], f"{label}: query", MAX_TEXT_CHARS),
        "positive": _check_text(record["positive"], f"{label}: positive", MAX_DOCUMENT_CHARS),
    }
    if record.get("negative") not in (None, ""):
        item["negative"] = _check_text(record["negative"], f"{label}: negative", MAX_DOCUMENT_CHARS)
        if item["negative"] == item["positive"]:
            raise ValueError(f"{label}: negative equals positive")
    if "intent" in record:
        item["intent"] = str(record["intent"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a query–positive dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, query, positive} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    queries: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        queries.add(item["query"].lower())
        checked.append(item)
    docs = documents(checked)
    if len(docs) < MIN_DOCUMENTS:
        raise ValueError(f"a dataset needs at least {MIN_DOCUMENTS} distinct positives; found {len(docs)}")
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_queries": len(queries),
        "n_documents": len(docs),
        "query_chars": {
            "min": min(len(r["query"]) for r in checked),
            "max": max(len(r["query"]) for r in checked),
        },
        "with_negative": sum("negative" in r for r in checked),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def documents(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted unique positive (and explicit negative) documents of a dataset — the candidates."""
    return sorted(
        {str(r["positive"]).strip() for r in records}
        | {str(r["negative"]).strip() for r in records if r.get("negative")}
    )


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["query"], r["positive"], r.get("negative", "")] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased query appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["query"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"query {record['query'][:60]!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating queries."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["query"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, query, positive[, negative]}` records from CSV (columns id, query, positive and an optional
    negative), a JSON array or JSONL."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "query", "positive"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        out = []
        for r in rows:
            item = {"id": r["id"], "query": r["query"], "positive": r["positive"]}
            if r.get("negative"):
                item["negative"] = r["negative"]
            out.append(item)
        return out
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "query", "positive", "negative"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "query": record["query"],
                    "positive": record["positive"],
                    "negative": record.get("negative", ""),
                }
            )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `11`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `97b0c614be4d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Qwen3EmbeddingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-embedding-0.6b",
  "modelId": "Qwen/Qwen3-Embedding-0.6B",
  "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
  "files": [
    {
      "path": "1_Pooling/config.json",
      "bytes": 313,
      "sha256": "37bf193fa101f19101bfad9c31d3eb0f786e247b7b1e5cb7f007d730eed1ddbd"
    },
    {
      "path": "README.md",
      "bytes": 17237,
      "sha256": "c34d9b7e5a267ad3fdd13227a253686bc90844ff4744a2a6a86c7c905e3d06f3"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "b5bf1f51fc45be473a54718cef92448d90a1be001bf9b9a44b8c7f10a19feaa9"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 215,
      "sha256": "10667c72ddb772627bf1780cb7f86af8e2ae0032b8c243c731172064105c6961"
    },
    {
      "path": "generation_config.json",
      "bytes": 117,
      "sha256": "28396d421a2108acce96383f6a7de78008f7f1b17f807958f3c14c51dbfb65fb"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191586416,
      "sha256": "0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd"
    },
    {
      "path": "modules.json",
      "bytes": 349,
      "sha256": "84e40c8e006c9b1d6c122e02cba9b02458120b5fb0c87b746c41e0207cf642cf"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11423705,
      "sha256": "def76fb086971c7867b829c23a26261e38d9d74e02139253b38aeb9df8b4b50a"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207487471
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Qwen3EmbeddingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the two pinned Banking77 CSV files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` checks the columns, row counts and the 77 intents of each member. `build_sample_dataset` turns every message into a `{id, query, positive}` pair whose positive is the intent name as words, drops repeated messages, and draws 8 training and 2 validation pairs per intent from the `train` member (disjoint messages) and 5 test pairs per intent from the `test` member by a seeded shuffle — the release's own partition, balanced over all 77 intents. `validate_dataset` then checks every record against the contract, `documents` lists the 77 retrieval candidates, `check_split_disjoint` asserts no message appears in two splits, and the training split is written to `outputs/qwen3_embedding_train.csv` in the shape BYOD expects. `INSTRUCTION` is the task description every query carries in later cells.

Look for: 10,003 + 3,080 raw rows, two digests, splits 616 / 154 / 385, 77 documents, and four refusal probes — a duplicate id, an empty query, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}
INSTRUCTION = 'Given a customer support message, retrieve the banking intent it expresses'  # @param {type:"string"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/banking77'))
    raw_rows = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
document_set = documents([*train_records, *val_records, *test_records])
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/qwen3_embedding_train.csv')
print({'data_source': data_source, 'instruction': INSTRUCTION, 'raw_rows': raw_rows, 'splits': disjoint, 'n_documents': len(document_set), 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_queries': manifest['unique_queries'], 'n_documents': manifest['n_documents'], 'query_chars': manifest['query_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': train_records[0], 'documents': document_set[:6] + ['...']})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty query': [{**train_records[0], 'query': '   '}, *train_records[1:8]],
    'missing field': [{'id': r['id'], 'query': r['query']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Embed through the inference contract

Before any adaptation, the embedding contract is exercised as it always was. `validate_inputs` applies exactly the checks `embed` applies — both route through the same private `_check_inputs` — so type, batch size 1..`MAX_BATCH`, non-empty text, the character ceiling, a `kind` in `KINDS` and a non-empty instruction are enforced identically; it returns an input manifest for the 77 documents with the three probe queries recorded under `query_manifest`, and a deliberately oversized batch is validated too and its rejection recorded as a finding. `embed` returns one unit-norm 1024-d vector per text with `n_tokens` and `truncated` flags; queries carry the instruction prefix, documents do not. The three test queries are ranked against the 77 documents by cosine and their top-3 documents printed beside the gold intent — a qualitative look before any metric is read; **cosine is a similarity, not a probability**, and no threshold ships. The frozen top-3 lists are kept as the *before* column for Section 9.

In [ ]:
import time

probe_records = test_records[:3]
probe_queries = [r['query'] for r in probe_records]
print({'ceilings': {'MAX_BATCH': MAX_BATCH, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'EMBEDDING_DIM': EMBEDDING_DIM, 'MAX_TRAIN_TOKENS': MAX_TRAIN_TOKENS, 'KINDS': KINDS}})
input_manifest = validate_inputs(document_set[:MAX_BATCH], 'document', names=[f'doc-{i:02d}' for i in range(min(len(document_set), MAX_BATCH))])
input_manifest['query_manifest'] = validate_inputs(probe_queries, 'query', INSTRUCTION, names=[r['id'] for r in probe_records])
try:
    validate_inputs(['probe'] * (MAX_BATCH + 1))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-batch-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/qwen3_embedding_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)

def top_documents(vectors, doc_vectors, k=3):
    scores = np.asarray(vectors, dtype=np.float32) @ np.asarray(doc_vectors, dtype=np.float32).T
    return [[(document_set[j], round(float(row[j]), 4)) for j in np.argsort(-row, kind='stable')[:k]] for row in scores]

started = time.perf_counter()
doc_rows = []
for start in range(0, len(document_set), MAX_BATCH):
    doc_rows.extend(pipe.embed(document_set[start:start + MAX_BATCH], kind='document')['embeddings'])
document_seconds = round(time.perf_counter() - started, 3)
started = time.perf_counter()
query_result = pipe.embed(probe_queries, kind='query', instruction=INSTRUCTION)
query_seconds = round(time.perf_counter() - started, 3)
doc_vectors = np.asarray(doc_rows, dtype=np.float32)
query_vectors = np.asarray(query_result['embeddings'], dtype=np.float32)
checks = {
    'one_vector_per_text': doc_vectors.shape == (len(document_set), EMBEDDING_DIM) and query_vectors.shape == (3, EMBEDDING_DIM),
    'unit_norm': bool(np.allclose(np.linalg.norm(doc_vectors, axis=1), 1.0, atol=1e-4)) and bool(np.allclose(np.linalg.norm(query_vectors, axis=1), 1.0, atol=1e-4)),
    'contract_fields': query_result['dim'] == EMBEDDING_DIM and query_result['pooling'] == POOLING and query_result['normalized'] is True and query_result['kind'] == 'query' and query_result['instruction'] == INSTRUCTION,
    'nothing_truncated': not any(query_result['truncated']),
    'all_values_finite': bool(np.isfinite(doc_vectors).all()) and bool(np.isfinite(query_vectors).all()),
}
if not all(checks.values()):
    raise RuntimeError(f'embed output failed a sanity check: {checks}')
before = dict(zip([r['id'] for r in probe_records], top_documents(query_vectors, doc_vectors), strict=True))
for record in probe_records:
    print({'id': record['id'], 'query': record['query'][:80], 'gold': record['positive'], 'frozen_top3': before[record['id']]})
print({'n_tokens': query_result['n_tokens'], 'seconds': {'documents': document_seconds, 'queries': query_seconds}, 'checks': checks, 'findings': len(input_manifest['findings']), 'score_semantics': 'cosine between unit vectors; a similarity, not a probability; no threshold shipped'})

## 6. The random floor, the lexical baseline and the frozen model on the test split

Three numbers frame the adaptation, all over the same 77-document set. The **random floor** is what a uniformly random ranking achieves in expectation (recall@1 = 1 / 77, MRR ≈ 0.064). The **lexical baseline** ranks the documents for each query by Jaccard overlap of lower-cased tokens — what a system with no model gets from shared words such as *card* or *pin*. `pipe.evaluate` embeds the 77 documents once and every test query with `INSTRUCTION`, ranks by cosine, and reads the rank of each query's own positive; ties are counted against the positive. Look for the frozen embedder well above both — the build record saw recall@1 in the sixties and MRR in the seventies — and read `median_rank` beside the means. About half a minute on CPU.

In [ ]:
def brief(m):
    return {k: round(m[k], 4) for k in ('recall@1', 'recall@5', 'recall@10', 'mrr')} | {'median_rank': m.get('median_rank')}

floor = random_floor(len(document_set))
print({'random_floor': {k: round(floor[k], 4) for k in ('recall@1', 'recall@5', 'recall@10', 'mrr')}, 'baseline': floor['baseline']})
t0 = time.perf_counter()
baseline_lexical = pipe.lexical_baseline(test_records, document_set)
print({'lexical_baseline': brief(baseline_lexical), 'baseline': baseline_lexical['baseline'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, instruction=INSTRUCTION, candidates=document_set)
print({'frozen_model_test': brief(frozen_test), 'n_queries': frozen_test['n_queries'], 'n_documents': frozen_test['n_documents'], 'verdict': frozen_test['verdict'], 'adapted': frozen_test['adapted'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
assert frozen_test['n_documents'] == baseline_lexical['n_documents'] and frozen_test['mrr'] > floor['mrr']

## 7. Bounded contrastive fine-tuning

`pipe.adapt` trains only the last `TRAINABLE_LAYERS` decoder layers — two by default, 31,461,888 of 595,776,512 parameters; the token embeddings, the earlier layers and the final norm stay frozen — with an **InfoNCE** loss: each batch embeds its queries (with `INSTRUCTION`) and the unique documents among its positives in one forward pass, and every query must pick its own positive out of the batch's documents by cosine at `TEMPERATURE` — the other queries' positives are the negatives. AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler; texts are truncated to `MAX_TRAIN_TOKENS` (64) **during training only**. Epoch 0 records the frozen model's validation retrieval metrics over the validation document set; every epoch is scored the same way, and the epoch with the highest validation MRR is kept.

Watch validation recall@1 climb by ten points or so over two epochs while recall@5 passes 95 % (about 47 s of training plus a validation pass per epoch on CPU). The build record's sweep on this sample: two layers at 2e-5 reached recall@1 77.1 %, four layers at 2e-5 82.1 % with a 252 MB adapter, two layers at 5e-5 80.3 % with a 126 MB adapter — the default.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}
TEMPERATURE = 0.05  # @param {type:"number"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_recall@1'] = round(entry['val']['recall@1'], 4)
        row['val_recall@5'] = round(entry['val']['recall@5'], 4)
        row['val_mrr'] = round(entry['val']['mrr'], 4)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, instruction=INSTRUCTION, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_layers=TRAINABLE_LAYERS, temperature=TEMPERATURE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'objective': adapt_result['objective'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'train_documents': adapt_result['n_train_documents'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no message in it appears in the training or validation splits. The adapted embedder is scored exactly as the frozen one was in Section 6 — same queries, same 77 documents, same instruction — and the four rows are put side by side. Look for recall@1 up by ten points or more and MRR up by about a tenth; the cell asserts the adapted MRR is above the frozen MRR. 385 queries from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on Banking77 intents says nothing about your retrieval task until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, instruction=INSTRUCTION, candidates=document_set)
adapted_val = pipe.evaluate(val_records, instruction=INSTRUCTION, candidates=documents(val_records))
comparison = {
    metric: {'random_floor': round(floor[metric], 4), 'lexical': round(baseline_lexical[metric], 4), 'frozen': round(frozen_test[metric], 4), 'adapted': round(adapted_test[metric], 4)}
    for metric in ('recall@1', 'recall@5', 'recall@10', 'mrr')
}
comparison['median_rank'] = {'lexical': baseline_lexical['median_rank'], 'frozen': frozen_test['median_rank'], 'adapted': adapted_test['median_rank']}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('recall@1', 'recall@5', 'recall@10', 'mrr')}
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'instruction': INSTRUCTION,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'n_documents': len(document_set),
    'baselines': {'random_floor': floor, 'lexical': baseline_lexical},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/qwen3_embedding_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['mrr'] > frozen_test['mrr']
print({'report': 'outputs/qwen3_embedding_evaluation_report.json'})

## 9. Retrieve before and after, export the adapter and reload it

The three test queries embedded by the frozen model in Section 5 are embedded again by the adapted model through the same `embed` contract, ranked against the re-embedded 77 documents, and their top-3 lists printed side by side with the gold intent. Read them as observations: the metric is Section 8, and the adapter moves the last decoder layers so **every vector changes** — each document's cosine to its frozen self is printed too. The per-batch `evaluation_report` helper — the inference-stage helper — is written for the probe queries and stays `not-measurable`, because a batch of vectors has no metric without a labelled set; `pipe.evaluate` is that labelled evaluation.

`pipe.save_artifact` writes the trained tensors — the last two decoder layers, about 126 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the instruction it was trained with, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `Qwen3EmbeddingPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not a decoder-layer tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical query vectors and an identical test MRR (VER4).

In [ ]:
import csv
import shutil

doc_rows_after = []
for start in range(0, len(document_set), MAX_BATCH):
    doc_rows_after.extend(pipe.embed(document_set[start:start + MAX_BATCH], kind='document')['embeddings'])
doc_vectors_after = np.asarray(doc_rows_after, dtype=np.float32)
query_vectors_after = np.asarray(pipe.embed(probe_queries, kind='query', instruction=INSTRUCTION)['embeddings'], dtype=np.float32)
after = dict(zip([r['id'] for r in probe_records], top_documents(query_vectors_after, doc_vectors_after), strict=True))
rows = []
for record in probe_records:
    rows.append({'id': record['id'], 'query': record['query'], 'gold': record['positive'], 'frozen_top3': ' | '.join(d for d, _s in before[record['id']]), 'adapted_top3': ' | '.join(d for d, _s in after[record['id']]), 'gold_rank_frozen': next((i + 1 for i, (d, _s) in enumerate(before[record['id']]) if d == record['positive']), None), 'gold_rank_adapted': next((i + 1 for i, (d, _s) in enumerate(after[record['id']]) if d == record['positive']), None)})
    print({k: rows[-1][k] for k in ('id', 'gold', 'frozen_top3', 'adapted_top3', 'gold_rank_frozen', 'gold_rank_adapted')})
document_shift = {'self_cosine_min': round(float(np.min(np.sum(doc_vectors * doc_vectors_after, axis=1))), 4), 'self_cosine_median': round(float(np.median(np.sum(doc_vectors * doc_vectors_after, axis=1))), 4)}
single_report = evaluation_report(pipe.embed(probe_queries, kind='query', instruction=INSTRUCTION), sample_kind='three Banking77 test queries' if not USE_BYOD else 'three BYOD test queries')
print({'document_shift': document_shift, 'batch_report_verdict': single_report['verdict'], 'probes_with_changed_top3': sum(r['frozen_top3'] != r['adapted_top3'] for r in rows), 'of': len(rows)})
with open('outputs/qwen3_embedding_retrieval.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/qwen3_embedding_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'qwen3_embedding', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'instruction': artifact_manifest['adapter']['instruction'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = Qwen3EmbeddingPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_queries = np.asarray(reloaded.embed(probe_queries, kind='query', instruction=INSTRUCTION)['embeddings'], dtype=np.float32)
reloaded_test = reloaded.evaluate(test_records[:77], instruction=INSTRUCTION, candidates=document_set)
in_memory_test = pipe.evaluate(test_records[:77], instruction=INSTRUCTION, candidates=document_set)
parity = {'query_vectors_identical': bool(np.array_equal(query_vectors_after, reloaded_queries)), 'mrr_in_memory': round(in_memory_test['mrr'], 6), 'mrr_reloaded': round(reloaded_test['mrr'], 6)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['query_vectors_identical'] and abs(in_memory_test['mrr'] - reloaded_test['mrr']) < 1e-9

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'instruction': INSTRUCTION,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'probe_queries': probe_queries, 'n_tokens': query_result['n_tokens'], 'seconds': {'documents': document_seconds, 'queries': query_seconds}},
    'comparison': comparison,
    'retrieval_before_after': rows,
    'document_shift': document_shift,
    'batch_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'bfloat16' if pipe.device.startswith('cuda') else 'float32'},
}
with open('outputs/qwen3_embedding_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen embedder already retrieves the right intent phrase for well over half of unseen messages (recall@1 in the sixties against a lexical baseline in the thirties and a random floor of 1.3 %), and a bounded contrastive fine-tuning of the last two decoder layers on 616 pairs lifts held-out recall@1 by more than ten points and MRR by about a tenth in a few minutes on CPU, with a 126 MB adapter that reloads to identical vectors. That is the claim: the adaptation contract can adapt the embedder to a retrieval task end to end on a real labelled corpus, and the numbers it produces are read against a random floor, a lexical baseline and the frozen model rather than in isolation.

The test split is 385 queries against 77 two-word documents from one seeded split of one corpus with no dispersion estimate; recall@k and MRR say whether the gold document ranks first, not whether the vectors are good for any other use. The adapter changes the last layers, which every input shares, so every vector shifts (Section 9 prints each document's cosine to its frozen self) and cosine values from the adapted model are not comparable to the frozen model's; nothing here measures the effect on other tasks. On CUDA the model runs in bfloat16 and the recorded float32 CPU numbers will not reproduce to the last digit.

Three things to carry to real data. **Floors first:** the random floor and the lexical baseline on *your* documents are the numbers to read before any embedder's; a lexical baseline near the frozen model means the task is mostly keyword matching. **Leakage:** de-duplicate queries across splits (the contract does this case-insensitively) and split by user or session when your queries come from one. **Documents:** the document set here is the label vocabulary; a real retrieval task has many more documents than intents and needs hard negatives beyond the in-batch ones this contract uses.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled corpus, validate the demonstrated dataset contract without leakage, execute the embedding contract and a bounded contrastive fine-tuning, evaluate by retrieval metrics against a random floor, a lexical baseline and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, embedding quality on any other task, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 1` and watch the gain shrink; set `TEMPERATURE = 0.2` and read whether the softer contrast learns as fast; set `EPOCHS = 4` and watch whether validation MRR keeps rising or turns (the best epoch is kept either way); change `INSTRUCTION` and re-read the frozen numbers — the model is instruction-aware; or bring your own pairs through BYOD and read the lexical baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/qwen3-embedding-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/qwen3-embedding-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/qwen3-embedding-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
- Upstream code: https://github.com/QwenLM/Qwen3-Embedding
- Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models (2025): https://arxiv.org/abs/2506.05176
- Efficient Intent Detection with Dual Sentence Encoders (Casanueva et al., 2020; Banking77, CC BY 4.0): https://arxiv.org/abs/2003.04807
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)